[目录](./table_of_contents_cn.ipynb)

# 最小二乘滤波器

In [1]:
%matplotlib inline

In [2]:
#format the book
import book_format
book_format.set_style()

## 简介

**作者注**：这部分内容是从 g-h 那一章中截取下来的，因为它放在那里并不合适。这一章还不适合阅读！我还没有写完它。

在那一章的开头附近，我使用 `numpy.polyfit()` 对体重测量值拟合了一条直线。它用“最小二乘拟合”把一个 n 次多项式拟合到数据上。这与 g-h 滤波器有什么不同呢？

这要视情况而定。我们最终会了解到，在某些条件下，从最小二乘拟合的角度看，卡尔曼滤波器是最优的。然而，`polyfit()` 是把多项式而不是任意曲线拟合到数据上，其做法是使下面这个公式的值最小：

$$E = \sum_{j=0}^k |p(x_j) - y_j|^2$$

我假设我的体重增长是恒定的，每天 1 磅，所以当我尝试拟合 $n=1$ 的多项式（也就是一条直线）时，结果与实际的体重增长非常吻合。但当然，没有人会一直只增重或只减重。我们的体重会波动。对更长的数据序列使用 'polyfit()' 会得到很差的结果。相比之下，g-h 滤波器会对变化率的改变作出反应——$h$ 这一项控制滤波器对这些变化的反应速度。如果我们增重，保持一段时间，然后减重，滤波器会自动跟踪这一变化。除非增重和减重能被某个多项式很好地表示，否则 'polyfit()' 做不到这一点。

即使数据符合 *n* 次多项式，这种形式的滤波器的另一个优势是它是*递归的*。也就是说，我们只需要知道上一个时间段的估计值和变化率，就能计算出这一时间段的估计值。相反，如果你深入 `polyfit()` 的实现，就会发现它需要全部数据才能给出答案。因此，像 `polyfit()` 这样的算法并不适合实时数据滤波。在 60 年代卡尔曼滤波器被发明的时候，计算机非常慢，内存也极其有限。它们完全无法存储诸如飞机惯性导航系统产生的成千上万个读数，也无法在提供准确、最新的导航信息所需的短时间内处理所有这些数据。


直到 20 世纪中叶，这类滤波一直使用各种形式的最小二乘估计。例如，NASA 的阿波罗计划有一个地面网络，用来跟踪指令服务舱（CSM）和登月舱（LM）。他们在许多分钟内进行测量，把数据成批汇集，然后缓慢地计算出答案。1960 年，NASA 艾姆斯研究中心的 Stanley Schmidt 认识到鲁道夫·卡尔曼那篇开创性论文的价值，并邀请他到艾姆斯访问。Schmidt 把卡尔曼的工作应用到 CSM 和 LM 的机载导航系统上，并称之为“卡尔曼滤波器”。[1] 此后不久，世界便转向了这种更快的递归滤波器。

卡尔曼滤波器只需要存储上一个估计值和少数几个相关参数，只需相对少量的计算就能生成下一个估计值。如今我们拥有如此多的内存和处理能力，这一优势变得没那么重要了，但在当时，卡尔曼滤波器是一项重大突破，不仅因为它的数学性质，还因为它能（勉强）在当时的硬件上运行。

这个主题比这段简短的讨论所显示的要深入得多。我们在全书中还会多次讨论这些话题。